# Experiment 1: Pix2Pix Baseline

**Architecture:** U-Net Generator + PatchGAN Discriminator (70x70 receptive field)  
**Task:** Given an empty room image + placement mask, generate the furnished room  
**Input:** 4 channels (room RGB + binary mask)  
**Output:** 3 channels (furnished room RGB)  
**Loss:** LSGAN + 100 * L1  

This is the simplest useful baseline for conditional image generation with paired data.  
It validates the full pipeline (data loading, training, evaluation) before moving to more complex architectures.

In [1]:
!pip install lpips torchmetrics -q

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.0 MB/s eta 0:00:0000:01
Using device: cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# ---- Paths (adjust to match your Google Drive layout) ----
DATA_ROOT = '/content/drive/MyDrive/693-project/data/processed'
CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')
CHECKPOINT_DIR = '/content/drive/MyDrive/693-project/checkpoints/pix2pix'
SAMPLE_DIR = '/content/drive/MyDrive/693-project/samples/pix2pix'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

Mounted at /content/drive


In [4]:
# ---- Hyperparameters ----
IMG_SIZE = 256
BATCH_SIZE = 8
LR = 2e-4
BETA1 = 0.5
BETA2 = 0.999
LAMBDA_L1 = 100
NUM_EPOCHS = 200
SAVE_EPOCH = 10
SAMPLE_EPOCH = 5

## Dataset

In [ ]:
class FurniturePlacementDataset(Dataset):
    def __init__(self, csv_path, split='train', img_size=256):
        df = pd.read_csv(csv_path)
        self.data = df[df['split'] == split].reset_index(drop=True)
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        room = Image.open(row['input_path']).convert('RGB')
        target = Image.open(row['target_path']).convert('RGB')

        orig_w, orig_h = room.size

        room = self.transform(room)
        target = self.transform(target)

        # Reconstruct binary mask from bbox, scaled to img_size
        mask = torch.zeros(1, self.img_size, self.img_size)
        x1 = int(row['bbox_x1'] * self.img_size / orig_w)
        y1 = int(row['bbox_y1'] * self.img_size / orig_h)
        x2 = int(row['bbox_x2'] * self.img_size / orig_w)
        y2 = int(row['bbox_y2'] * self.img_size / orig_h)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(self.img_size, x2), min(self.img_size, y2)
        mask[:, y1:y2, x1:x2] = 1.0

        condition = torch.cat([room, mask], dim=0)

        return {
            'condition': condition,
            'target': target,
            'room': room,
            'mask': mask,
        }

In [ ]:
train_dataset = FurniturePlacementDataset(CSV_PATH, split='train', img_size=IMG_SIZE)
val_dataset = FurniturePlacementDataset(CSV_PATH, split='val', img_size=IMG_SIZE)
test_dataset = FurniturePlacementDataset(CSV_PATH, split='test', img_size=IMG_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

# Visualize a batch
batch = next(iter(train_loader))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    room = batch['room'][i] * 0.5 + 0.5
    target = batch['target'][i] * 0.5 + 0.5
    mask = batch['mask'][i]
    axes[0, i].imshow(room.permute(1, 2, 0).numpy())
    axes[0, i].set_title('Empty room')
    axes[0, i].axis('off')
    axes[1, i].imshow(mask.squeeze().numpy(), cmap='gray')
    axes[1, i].set_title('Placement mask')
    axes[1, i].axis('off')
    axes[2, i].imshow(target.permute(1, 2, 0).numpy())
    axes[2, i].set_title('Target (furnished)')
    axes[2, i].axis('off')
plt.tight_layout()
plt.show()

## U-Net Generator

In [ ]:
class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.model(x)
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """U-Net generator for 256x256 images.
    Encoder: 256->128->64->32->16->8->4->2->1
    Decoder mirrors with skip connections."""

    def __init__(self, in_channels=4, out_channels=3):
        super().__init__()
        self.down1 = UNetDown(in_channels, 64, normalize=False)
        self.down2 = UNetDown(64, 128)
        self.down3 = UNetDown(128, 256)
        self.down4 = UNetDown(256, 512, dropout=0.5)
        self.down5 = UNetDown(512, 512, dropout=0.5)
        self.down6 = UNetDown(512, 512, dropout=0.5)
        self.down7 = UNetDown(512, 512, dropout=0.5)
        self.down8 = UNetDown(512, 512, normalize=False, dropout=0.5)

        self.up1 = UNetUp(512, 512, dropout=0.5)
        self.up2 = UNetUp(1024, 512, dropout=0.5)
        self.up3 = UNetUp(1024, 512, dropout=0.5)
        self.up4 = UNetUp(1024, 512, dropout=0.5)
        self.up5 = UNetUp(1024, 256)
        self.up6 = UNetUp(512, 128)
        self.up7 = UNetUp(256, 64)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.down1(x)   # 128
        d2 = self.down2(d1)  # 64
        d3 = self.down3(d2)  # 32
        d4 = self.down4(d3)  # 16
        d5 = self.down5(d4)  # 8
        d6 = self.down6(d5)  # 4
        d7 = self.down7(d6)  # 2
        d8 = self.down8(d7)  # 1

        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)

        return self.final(u7)

## PatchGAN Discriminator (70x70 receptive field)

In [ ]:
class PatchGANDiscriminator(nn.Module):
    """70x70 PatchGAN. Output is a 30x30 map for 256x256 input.
    Each output pixel classifies a 70x70 receptive field patch."""

    def __init__(self, in_channels=7):
        super().__init__()
        self.model = nn.Sequential(
            # C64, no norm
            nn.Conv2d(in_channels, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            # C128
            nn.Conv2d(64, 128, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # C256
            nn.Conv2d(128, 256, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # C512 stride=1
            nn.Conv2d(256, 512, 4, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # Output stride=1
            nn.Conv2d(512, 1, 4, stride=1, padding=1),
        )

    def forward(self, condition, target):
        x = torch.cat([condition, target], dim=1)
        return self.model(x)

## Initialize Models

In [ ]:
def init_weights(m):
    classname = m.__class__.__name__
    if hasattr(m, 'weight') and ('Conv' in classname):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif 'Norm' in classname and m.weight is not None:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


generator = UNetGenerator(in_channels=4, out_channels=3).to(device)
discriminator = PatchGANDiscriminator(in_channels=7).to(device)

generator.apply(init_weights)
discriminator.apply(init_weights)

criterion_gan = nn.MSELoss()   # LSGAN
criterion_l1 = nn.L1Loss()

opt_g = optim.Adam(generator.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_d = optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA1, BETA2))

# Linear LR decay over the last 50% of training
def lambda_rule(epoch):
    decay_start = NUM_EPOCHS // 2
    if epoch < decay_start:
        return 1.0
    return 1.0 - (epoch - decay_start) / (NUM_EPOCHS - decay_start)

sched_g = optim.lr_scheduler.LambdaLR(opt_g, lr_lambda=lambda_rule)
sched_d = optim.lr_scheduler.LambdaLR(opt_d, lr_lambda=lambda_rule)

n_g = sum(p.numel() for p in generator.parameters())
n_d = sum(p.numel() for p in discriminator.parameters())
print(f'Generator params:     {n_g:,}')
print(f'Discriminator params: {n_d:,}')

## Training

In [ ]:
def save_samples(epoch, generator, loader, sample_dir, n=4):
    generator.eval()
    batch = next(iter(loader))
    condition = batch['condition'][:n].to(device)
    target = batch['target'][:n]
    room = batch['room'][:n]
    mask = batch['mask'][:n]

    with torch.no_grad():
        fake = generator(condition).cpu()

    fake_vis = fake * 0.5 + 0.5
    target_vis = target * 0.5 + 0.5
    room_vis = room * 0.5 + 0.5

    fig, axes = plt.subplots(4, n, figsize=(4 * n, 16))
    for i in range(n):
        axes[0, i].imshow(room_vis[i].permute(1, 2, 0).numpy())
        axes[0, i].set_title('Input')
        axes[0, i].axis('off')
        axes[1, i].imshow(mask[i].squeeze().numpy(), cmap='gray')
        axes[1, i].set_title('Mask')
        axes[1, i].axis('off')
        axes[2, i].imshow(fake_vis[i].permute(1, 2, 0).clamp(0, 1).numpy())
        axes[2, i].set_title('Generated')
        axes[2, i].axis('off')
        axes[3, i].imshow(target_vis[i].permute(1, 2, 0).numpy())
        axes[3, i].set_title('Ground Truth')
        axes[3, i].axis('off')

    plt.suptitle(f'Epoch {epoch}', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(sample_dir, f'epoch_{epoch:04d}.png'), dpi=100)
    plt.show()
    plt.close()
    generator.train()

In [ ]:
history = {'g_loss': [], 'd_loss': [], 'l1_loss': [], 'val_l1': []}

for epoch in range(1, NUM_EPOCHS + 1):
    generator.train()
    discriminator.train()

    epoch_g, epoch_d, epoch_l1 = 0.0, 0.0, 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for batch in pbar:
        condition = batch['condition'].to(device)
        target = batch['target'].to(device)

        fake = generator(condition)

        # ---- Train Discriminator ----
        opt_d.zero_grad()

        pred_real = discriminator(condition, target)
        pred_fake = discriminator(condition, fake.detach())

        real_label = torch.ones_like(pred_real, device=device)
        fake_label = torch.zeros_like(pred_fake, device=device)

        loss_d = 0.5 * (criterion_gan(pred_real, real_label) +
                        criterion_gan(pred_fake, fake_label))
        loss_d.backward()
        opt_d.step()

        # ---- Train Generator ----
        opt_g.zero_grad()

        pred_fake_for_g = discriminator(condition, fake)
        loss_g_gan = criterion_gan(pred_fake_for_g, real_label)
        loss_g_l1 = criterion_l1(fake, target)
        loss_g = loss_g_gan + LAMBDA_L1 * loss_g_l1

        loss_g.backward()
        opt_g.step()

        epoch_g += loss_g.item()
        epoch_d += loss_d.item()
        epoch_l1 += loss_g_l1.item()

        pbar.set_postfix({'G': f'{loss_g.item():.4f}', 'D': f'{loss_d.item():.4f}'})

    n_batches = len(train_loader)
    history['g_loss'].append(epoch_g / n_batches)
    history['d_loss'].append(epoch_d / n_batches)
    history['l1_loss'].append(epoch_l1 / n_batches)

    # Validation L1
    generator.eval()
    val_l1 = 0.0
    with torch.no_grad():
        for vb in val_loader:
            vc = vb['condition'].to(device)
            vt = vb['target'].to(device)
            vf = generator(vc)
            val_l1 += criterion_l1(vf, vt).item()
    history['val_l1'].append(val_l1 / max(len(val_loader), 1))

    sched_g.step()
    sched_d.step()

    print(f'Epoch {epoch} | G: {history["g_loss"][-1]:.4f} | '
          f'D: {history["d_loss"][-1]:.4f} | '
          f'L1: {history["l1_loss"][-1]:.4f} | '
          f'Val L1: {history["val_l1"][-1]:.4f}')

    if epoch % SAMPLE_EPOCH == 0:
        save_samples(epoch, generator, val_loader, SAMPLE_DIR)

    if epoch % SAVE_EPOCH == 0:
        torch.save({
            'epoch': epoch,
            'generator': generator.state_dict(),
            'discriminator': discriminator.state_dict(),
            'opt_g': opt_g.state_dict(),
            'opt_d': opt_d.state_dict(),
            'history': history,
        }, os.path.join(CHECKPOINT_DIR, f'checkpoint_{epoch:04d}.pt'))

# Save final model
torch.save({
    'epoch': NUM_EPOCHS,
    'generator': generator.state_dict(),
    'discriminator': discriminator.state_dict(),
    'history': history,
}, os.path.join(CHECKPOINT_DIR, 'final_model.pt'))
print('Training complete.')

## Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(history['g_loss'])
axes[0].set_title('Generator Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['d_loss'])
axes[1].set_title('Discriminator Loss')
axes[1].set_xlabel('Epoch')

axes[2].plot(history['l1_loss'])
axes[2].set_title('Train L1')
axes[2].set_xlabel('Epoch')

axes[3].plot(history['val_l1'])
axes[3].set_title('Val L1')
axes[3].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'loss_curves.png'), dpi=150)
plt.show()

## Evaluation

In [ ]:
import lpips
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM
from torchmetrics.image import PeakSignalNoiseRatio as PSNR

lpips_fn = lpips.LPIPS(net='alex').to(device)
ssim_fn = SSIM(data_range=1.0).to(device)
psnr_fn = PSNR(data_range=1.0).to(device)

generator.eval()
all_lpips, all_ssim, all_psnr = [], [], []
bg_psnr_list, bg_ssim_list = [], []

# Directories for FID computation
fid_real_dir = os.path.join(SAMPLE_DIR, 'fid_real')
fid_fake_dir = os.path.join(SAMPLE_DIR, 'fid_fake')
os.makedirs(fid_real_dir, exist_ok=True)
os.makedirs(fid_fake_dir, exist_ok=True)

img_idx = 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        condition = batch['condition'].to(device)
        target = batch['target'].to(device)
        mask = batch['mask'].to(device)

        fake = generator(condition)

        fake_01 = fake * 0.5 + 0.5
        target_01 = target * 0.5 + 0.5

        # Per-image metrics
        all_lpips.append(lpips_fn(fake, target).mean().item())
        all_ssim.append(ssim_fn(fake_01, target_01).item())
        all_psnr.append(psnr_fn(fake_01, target_01).item())

        # Background preservation (non-furniture region)
        bg_mask = 1.0 - mask
        if bg_mask.sum() > 0:
            bg_fake = fake_01 * bg_mask
            bg_real = target_01 * bg_mask
            bg_psnr_list.append(psnr_fn(bg_fake, bg_real).item())
            bg_ssim_list.append(ssim_fn(bg_fake, bg_real).item())

        # Save images for FID
        for j in range(fake_01.size(0)):
            real_img = transforms.ToPILImage()(target_01[j].cpu())
            fake_img = transforms.ToPILImage()(fake_01[j].cpu().clamp(0, 1))
            real_img.save(os.path.join(fid_real_dir, f'{img_idx:05d}.png'))
            fake_img.save(os.path.join(fid_fake_dir, f'{img_idx:05d}.png'))
            img_idx += 1

print('\n===== Pix2Pix Test Metrics =====')
print(f'  LPIPS:    {np.mean(all_lpips):.4f}  (lower is better)')
print(f'  SSIM:     {np.mean(all_ssim):.4f}  (higher is better)')
print(f'  PSNR:     {np.mean(all_psnr):.2f} dB  (higher is better)')
if bg_psnr_list:
    print(f'  BG-PSNR:  {np.mean(bg_psnr_list):.2f} dB')
    print(f'  BG-SSIM:  {np.mean(bg_ssim_list):.4f}')
print('================================')

In [ ]:
# Compute FID (requires pytorch-fid)
!python -m pytorch_fid {fid_real_dir} {fid_fake_dir}

## Final Results Visualization

In [ ]:
generator.eval()
batch = next(iter(test_loader))
n_show = min(8, batch['condition'].size(0))
condition = batch['condition'][:n_show].to(device)
target = batch['target'][:n_show]
room = batch['room'][:n_show]
mask = batch['mask'][:n_show]

with torch.no_grad():
    fake = generator(condition).cpu()

fake_vis = (fake * 0.5 + 0.5).clamp(0, 1)
target_vis = target * 0.5 + 0.5
room_vis = room * 0.5 + 0.5

fig, axes = plt.subplots(4, n_show, figsize=(4 * n_show, 16))
row_labels = ['Empty Room', 'Mask', 'Generated', 'Ground Truth']
for i in range(n_show):
    axes[0, i].imshow(room_vis[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(mask[i].squeeze().numpy(), cmap='gray')
    axes[1, i].axis('off')
    axes[2, i].imshow(fake_vis[i].permute(1, 2, 0).numpy())
    axes[2, i].axis('off')
    axes[3, i].imshow(target_vis[i].permute(1, 2, 0).numpy())
    axes[3, i].axis('off')

for ax, label in zip(axes[:, 0], row_labels):
    ax.set_ylabel(label, fontsize=14, rotation=90, labelpad=20)

plt.suptitle('Pix2Pix -- Test Set Results', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'final_results.png'), dpi=150, bbox_inches='tight')
plt.show()